# W2.6-D9R13-PALIGEMMA-FROZEN-EXTERNAL-GATE-PREP

Draft PR #60, PREP only; no execution by Codex. Reuse the same eight frozen synthetic-v1 object-localization cases as Qwen/Moondream. No C1/B2 qualification, new fixtures, runtime regeneration, policy, annotation or threshold change.

One visible T4; FP16; quantization NONE; one model load; eight native grounding calls; zero classification calls. Exact checkpoint/revision and D9R11 32-token greedy decoding are unchanged.

Native prompts are exactly `detect red square`, `detect green circle`, `detect yellow triangle`, `detect cyan rectangle`. The frozen manifest contains `Locate the ... .` queries; a closed four-entry lookup supplies bare labels without changing the manifest. No canonical probe JSON wrapper, punctuation or manual newline is passed to the processor. The native processor owns image/BOS tokens and the terminal newline. The parser accepts one loc box plus exactly one frozen label and EOS; /1024, no clamp or repair. A wrong label stays in raw diagnostics and fails closed.

Use unchanged `safeshift.protocol.gate.evaluate_gate`. Automatic failure is FAIL; otherwise the gate remains PENDING_REVIEW until explicit human giant-box review. This notebook reports EVIDENCE_COLLECTION_COMPLETE or STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED, never auto-promotes the model. Production classification remains PENDING_QUALIFICATION and does not block this gate.

D9R11 case grd_d hallucinated a red-square detection with no target. Its documentary evidence is preserved in the bundle and must inform the Research Lead's final decision; no extra case is added here.


In [ ]:
# Definitions only; no model/backend import or network request in this cell.
import os
import sys
if "torch" in sys.modules or "transformers" in sys.modules:
    raise RuntimeError("STOP: use a fresh notebook kernel before any backend import")
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import hashlib
import json
import math
import platform
import re
import shutil
import subprocess
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

BASE = "bcc7b7be62891c887e9509dd39688d686a1b6020"
BASE_SHA = "a3192ddefbc28fe2997190819744e8f4796a6d85"
QUAL_PLAN = "configs/pre_freeze/paligemma_external_gate.v1.json"
QUAL_HASH = "c3b5fb2a1df79f3476e3775fccefbe2f767edab453ba2a238742c4c5f89d4387"
MODEL = "google/paligemma-3b-mix-448"
REVISION = "ead2d9a35598cb89119af004f5d023b311d1c4a1"
PLAN_HASH = "4138071ff3fcbcf2cb16d456c01a2b3dbc42a628b0faa0d04dd6a635803b0ab7"
WORKING = Path("/kaggle/working")  # Venue mount, not a personal machine path.
WORK = WORKING / "safeshift_d9r13"
REPO = WORK / "runtime_checkout"
SETUP = WORK / "evidence"
ENV = WORK / "runtime_env"
PYTHON = ENV / "bin/python"
CACHE = "data/processed/paligemma_d9r13_hf_cache"
MANIFESTS = "data/processed/d9r13_manifests"
RUN_ID = "paligemma-d9r13-external-gate-01"  # Fixed attempt identity; never regenerate on failure.
RUN_REL = "data/processed/external_gate/w2_paligemma/" + RUN_ID
BUNDLE = WORKING / "d9r13_result_bundle"
ZIP = WORKING / "d9r13_result_bundle.zip"
OFFLINE = {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
           "HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_ENABLE_HF_TRANSFER": "0",
           "HF_HUB_DISABLE_XET": "1"}
SECRET_VALUES = []  # Memory only; never serialized or printed.
ONLINE_DONE = False
PHASE = "NOT_STARTED"
MAX_FILE = 4 * 1024**2
MAX_BUNDLE = 32 * 1024**2

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def digest(data):
    return hashlib.sha256(data).hexdigest()

def sanitize(text):
    for value in sorted(SECRET_VALUES, key=len, reverse=True):
        if value:
            text = text.replace(value, "[REDACTED]")
    text = re.sub(r"\b(?:hf_|ghp_|gho_|github_pat_)[A-Za-z0-9_]{16,}", "[REDACTED]", text)
    text = re.sub(r"(?i)(authorization\s*[:=]\s*)(?:bearer|basic)\s+\S+", r"\1[REDACTED]", text)
    text = re.sub(r"https?://[^\s/@]+:[^\s/@]+@", "https://[REDACTED]@", text)
    return text

def write_new(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = (json.dumps(value, indent=2, allow_nan=False) + "\n").encode()
    require(sanitize(data.decode()) == data.decode(), "STOP: secret-like content refused")
    with path.open("xb") as stream:
        stream.write(data)
        stream.flush()
        os.fsync(stream.fileno())

def child_env(offline=False):
    # Allowlist: never inherit notebook tokens, Python paths, Hub/Git credentials,
    # proxy credentials, or user pip settings into a child process.
    env = {key: os.environ[key] for key in
           ("PATH", "HOME", "LANG", "LC_ALL", "TMPDIR", "LD_LIBRARY_PATH") if key in os.environ}
    env.update(CUDA_DEVICE_ORDER="PCI_BUS_ID", CUDA_VISIBLE_DEVICES="0",
               PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1", PIP_CONFIG_FILE="/dev/null",
               HF_HOME=str(WORK / "private_hf_home"), HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_ENABLE_HF_TRANSFER="0", HF_HUB_DISABLE_XET="1",
               GIT_TERMINAL_PROMPT="0", GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL="/dev/null",
               UV_PYTHON_INSTALL_DIR=str(WORK / "managed_python"),
               UV_PYTHON_BIN_DIR=str(WORK / "python_bin"), UV_CACHE_DIR=str(WORK / "uv_cache"))
    if offline:
        env.update(OFFLINE)
    return env

def command(args, label, *, cwd=None, env=None, failure=None, check=True):
    print("START", label, flush=True)
    started = time.perf_counter()
    result = subprocess.run([str(a) for a in args], cwd=cwd,
                            env=env if env is not None else child_env(),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    # Sanitize before any disk write/display. Never log the environment.
    log = sanitize(result.stdout)
    require(len(log.encode()) <= MAX_FILE, "STOP: oversized log; no evidence truncation")
    with (SETUP / (label + ".log")).open("x", encoding="utf-8") as stream:
        stream.write(log)
    write_new(SETUP / (label + ".json"), {
        "returncode": result.returncode, "wall_seconds": time.perf_counter() - started})
    if check and result.returncode != 0:
        raise RuntimeError(failure or ("STOP: " + label + "; inspect sanitized log"))
    print("PASS" if result.returncode == 0 else "FAIL", label, flush=True)
    return result.returncode, log

def secret(name, required=True):
    value = None
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        pass  # Never propagate provider exceptions that could carry credentials.
    if not value:
        if required:
            raise RuntimeError("Create a Kaggle Secret named " + name + " and rerun.") from None
        return None
    SECRET_VALUES.append(value)
    return value

def clean_checkout():
    env = child_env(offline=True)
    for args, expected in ((["rev-parse", "HEAD"], BASE),
                           (["status", "--porcelain", "--untracked-files=all"], "")):
        value = subprocess.check_output(["git", *args], cwd=REPO, env=env, text=True).strip()
        require(value == expected, "STOP: exact clean execution checkout required")

def checked_file(root, relative):
    rel = PurePosixPath(relative)
    require(not rel.is_absolute() and ".." not in rel.parts and ":" not in relative
            and "\\" not in relative, "STOP: unsafe evidence path")
    path = root / relative
    require(path.resolve().is_relative_to(root.resolve()) and not path.is_symlink()
            and path.resolve() == path.absolute(), "STOP: evidence symlink/escape")
    require(path.is_file() and path.stat().st_size <= MAX_FILE, "STOP: missing/oversized evidence")
    return path

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def build_bundle(report):
    # Positive allowlist only. Never traverse the checkout/cache/environment.
    require(not BUNDLE.exists() and not ZIP.exists(), "STOP: bundle already exists; no overwrite")
    BUNDLE.mkdir()
    selected = []
    for path in sorted(SETUP.iterdir()):
        if path.suffix in {".json", ".log", ".py"}:
            selected.append((SETUP, path.name, "setup/" + path.name))
    manifests = REPO / MANIFESTS
    for name in ("provision.json", "verify_online.json", "verify_offline.json"):
        if (manifests / name).is_file():
            selected.append((manifests, name, "snapshot/" + name))
    if (REPO / QUAL_PLAN).is_file():
        selected.append((REPO, QUAL_PLAN, "plan/paligemma_external_gate.v1.json"))
    for relative in ("configs/pre_freeze/external_gate_cases.v1.json",
                     "configs/pre_freeze/external_gate_cases.v1.provenance.json",
                     "configs/pre_freeze/paligemma_interface_runtime_result.v1.json"):
        if (REPO / relative).is_file():
            selected.append((REPO, relative, "authority/" + Path(relative).name))
    runtime = REPO / RUN_REL
    if runtime.is_dir():
        allowed = {"summary.json", "summary.sha256.json", "gate_plan.json",
                   "fixture_manifest.json", "gate.json", "human_review_template.json"}
        allowed.update(f"{g}_{n}.{ext}" for g in "ABCD" for n in (1, 2) for ext in ("json", "png"))
        for path in sorted(runtime.rglob("*")):
            if not path.is_file():
                continue
            rel = path.relative_to(runtime).as_posix()
            if rel in allowed or re.fullmatch(r"[0-9a-f]{64}/(?:response\.raw|metadata\.json|result\.json)", rel):
                selected.append((runtime, rel, "runtime/" + rel))
    inventory, total = {}, 0
    for root, relative, target in selected:
        path = checked_file(root, relative)
        raw = path.read_bytes()
        # Logs were sanitized at capture. Raw/model evidence is NEVER rewritten.
        if path.suffix not in {".png", ".input"}:
            text = raw.decode("utf-8")
            require(sanitize(text) == text, "STOP: secret-like content in evidence; export refused")
        total += len(raw)
        require(total <= MAX_BUNDLE, "STOP: evidence bundle exceeds size limit")
        destination = BUNDLE / target
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("xb") as stream:
            stream.write(raw)
        inventory[target] = {"sha256": digest(raw), "size_bytes": len(raw)}
    write_new(BUNDLE / "review_summary.json", report)
    raw = (BUNDLE / "review_summary.json").read_bytes()
    inventory["review_summary.json"] = {"sha256": digest(raw), "size_bytes": len(raw)}
    write_new(BUNDLE / "checksums.json", inventory)
    with zipfile.ZipFile(ZIP, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(BUNDLE.rglob("*")):
            if path.is_file():
                archive.write(path, "d9r13_result_bundle/" + path.relative_to(BUNDLE).as_posix())
    print("BUNDLE:", ZIP)
    print("ZIP_SHA256:", digest(ZIP.read_bytes()))
    from IPython.display import FileLink, display
    display(FileLink(str(ZIP.relative_to(WORKING))))

def boundaries():
    return {"model": MODEL, "revision": REVISION, "base_sha": BASE_SHA,
            "execution_commit": BASE, "qualification_plan_path": QUAL_PLAN,
            "qualification_plan_sha256": QUAL_HASH, "run_id": RUN_ID,
            "qualification_decision": "RESEARCH_LEAD_REVIEW_REQUIRED",
            "real_runtime_status": "RUNTIME_SMOKE_PASS", "exact_runtime_verified": True,
            "grounding_status": "PENDING_QUALIFICATION",
            "classification_interface_status": "PENDING_QUALIFICATION",
            "external_gate_status": "PENDING_QUALIFICATION",
            "other_synthetic_execution": "NOT_RUN", "inspecsafe": "NOT_RUN", "inspecsafe_authorized": False,
            "promotion": "NO", "paligemma_role": "BACKUP_1", "primary_roster_count": 4,
            "protocol_freeze": "BLOCKED"}

def review_runtime(returncode):
    root = REPO / RUN_REL
    raw = checked_file(root, "summary.json").read_bytes()
    require(digest(raw) == read_json(checked_file(root, "summary.sha256.json"))["sha256"],
            "STOP: summary hash mismatch")
    summary = json.loads(raw)
    require(returncode == 0 and summary["status"] == "EVIDENCE_COLLECTION_COMPLETE",
            "STOP: harness failed; retain partial evidence without retry")
    qualification = read_json(REPO / QUAL_PLAN)
    require(digest((REPO / QUAL_PLAN).read_bytes()) == QUAL_HASH,
            "STOP: qualification plan changed")
    require(summary["plan_sha256"] == QUAL_HASH and summary["base_sha"] == BASE_SHA
            and summary["execution_commit"] == BASE and summary["model_id"] == MODEL
            and summary["revision"] == REVISION and summary["run_id"] == RUN_ID,
            "STOP: runtime identity mismatch")
    require(summary["model_load_count"] == 1 and summary["native_generate_calls"] == 8
            and summary["classification_calls"] == 0
            and summary["classification_interface_status"] == "PENDING_QUALIFICATION"
            and summary["promotion"] is False
            and [c["case_id"] for c in summary["calls"]] == qualification["case_ids"],
            "STOP: one-load/eight-call matrix mismatch")
    require(summary["preserved_status"] == qualification["preserved_status"]
            and summary["interpretation"] == qualification["interpretation"],
            "STOP: unauthorized status/interpretation change")
    require(summary["offline"]["owner_attested_internet_off"] is True
            and summary["offline"]["variables"] == OFFLINE
            and summary["offline"]["socket_denial"] is True
            and summary["software"] == plan["software"],
            "STOP: runtime software/offline mismatch")
    hardware = summary["hardware"]
    require(hardware["visible_gpu_count"] == 1 and hardware["device"] == "cuda:0"
            and hardware["gpu_name"] in {"T4", "Tesla T4", "NVIDIA T4"}
            and hardware["compute_capability"] == [7, 5]
            and hardware["cuda_runtime"] == "12.4"
            and 14 * 2**30 <= hardware["total_vram_bytes"] <= 16 * 2**30,
            "STOP: runtime hardware mismatch")
    snapshot = summary["snapshot"]
    offline = read_json(REPO / MANIFESTS / "verify_offline.json")
    online = read_json(REPO / MANIFESTS / "verify_online.json")
    provision = read_json(REPO / MANIFESTS / "provision.json")
    require(snapshot["local_bytes_verified"] is True and len(snapshot["files"]) == 14
            and snapshot["revision"] == REVISION
            and snapshot["files"] == offline["files"] == online["files"] == provision["files"],
            "STOP: snapshot manifests disagree")
    for relative, ref in summary["artifacts"].items():
        raw = checked_file(root, relative).read_bytes()
        require(digest(raw) == ref["sha256"] and len(raw) == ref["size_bytes"],
                "STOP: runtime artifact hash/size mismatch")
    seen = set()
    manifest = read_json(REPO / qualification["manifest"])
    for case, call in zip(manifest["cases"], summary["calls"]):
        require(call["result_path"] == case["case_id"] + ".json", "STOP: result path mismatch")
        result = read_json(checked_file(root, call["result_path"]))
        ref = call["raw"]
        prefix = RUN_REL + "/"
        require(ref["path"].startswith(prefix) and ref["path"] not in seen,
                "STOP: raw outside run or reused")
        seen.add(ref["path"])
        relative = ref["path"][len(prefix):]
        raw = checked_file(root, relative).read_bytes()
        require(digest(raw) == ref["sha256"] and len(raw) == ref["size_bytes"],
                "STOP: raw hash/size mismatch")
        metadata = read_json(checked_file(root, relative.rsplit("/", 1)[0] + "/metadata.json"))
        prompt = "detect " + qualification["query_to_label"][case["target_query"]]
        require(metadata["parse_status"] == "NOT_ATTEMPTED"
                and metadata["call_id"] == case["case_id"]
                and metadata["git_commit_sha"] == BASE
                and metadata["task_type"] == "grounding" and metadata["prompt"] == prompt
                and result["prompt"] == call["prompt"] == prompt and result["raw"] == ref
                and result["status"] == "EVIDENCE_COLLECTION_COMPLETE",
                "STOP: raw-before-parser provenance mismatch")
        for key, value in case.items():
            require(result["fixture"][key] == call["fixture"][key] == value,
                    "STOP: frozen case/GT changed")
        require(result["observation"]["native_text"] == json.loads(raw).get("decoded_with_special_tokens")
                and result["observation"] == call["observation"]
                and result["parser_status"] == call["parser_status"]
                and result["predicted_bbox"] == call["predicted_bbox"]
                and (result["predicted_bbox"] is not None) == (result["parser_status"] == "SUCCESS"),
                "STOP: native/fail-closed result mismatch")
        require(call["gate_result"] == next(c for c in summary["gate"]["cases"]
                                            if c["case_id"] == case["case_id"]),
                "STOP: gate metrics mismatch")
    require(summary["gate"] == read_json(checked_file(root, "gate.json"))
            and summary["paligemma_external_gate_status"] == summary["gate"]["status"]
            and summary["gate"]["status"] in {"FAIL", "PENDING_REVIEW"},
            "STOP: runtime cannot auto-PASS without human review")
    return {**boundaries(), "status": "EVIDENCE_COLLECTION_COMPLETE",
            "artifact_checksums_verified": True, "raw_before_parser": True,
            "model_load_count": 1, "native_generate_calls": 8, "cases": 8,
            "summary": "runtime/summary.json",
            "paligemma_external_gate_status": summary["paligemma_external_gate_status"],
            "negative_evidence": summary["negative_evidence"],
            "research_lead_decision": "REQUIRED; explicit human giant-box review; no promotion"}


## Online setup for the future authorized D9R13 run
This cell provisions the exact snapshot after isolated environment checks.
No model load occurs here. D9R13 prepares this cell but does not execute it.


In [ ]:
require(platform.system() == "Linux" and platform.machine() == "x86_64"
        and WORKING.is_dir(), "STOP: Kaggle Linux x86_64 venue required")
require(not WORK.exists() and not BUNDLE.exists() and not ZIP.exists(),
        "STOP: dedicated workspace/evidence already exists; no repair, overwrite or automatic retry")
# Missing credential stops before creating a workspace, so the prescribed secret
# setup and rerun is safe. Other failures retain their workspace for review.
hf_token = secret("HF_TOKEN")
WORK.mkdir()
SETUP.mkdir()
try:
    PHASE = "VENUE"
    _, physical = command(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap,driver_version",
                           "--format=csv,noheader,nounits"], "physical_gpu")
    rows = [[part.strip() for part in row.split(",")] for row in physical.strip().splitlines()]
    require(len(rows) in (1, 2), "STOP: expected one or two physical T4 GPUs")
    for index, name, memory, capability, driver in rows:
        require(name in {"T4", "Tesla T4", "NVIDIA T4"} and capability == "7.5"
                and 14 * 1024 <= float(memory) <= 16 * 1024, "STOP: physical GPU contract mismatch")
    write_new(SETUP / "venue.json", {"platform": platform.platform(), "physical_gpus": rows,
              "notebook_python": platform.python_version(), "free_disk_bytes": shutil.disk_usage(WORK).free,
              "cuda_visible_devices": "0", "cuda_device_order": "PCI_BUS_ID"})

    PHASE = "CHECKOUT"
    command(["git", "-c", "credential.helper=", "clone", "--no-checkout",
             "https://github.com/tantdna2/SafeShift.git", REPO], "clone",
            failure="STOP: anonymous public clone failed; preserve evidence for Research Lead review.")
    command(["git", "checkout", "--detach", BASE], "checkout", cwd=REPO)
    clean_checkout()
    command(["git", "merge-base", "--is-ancestor", BASE_SHA, BASE], "base_ancestry", cwd=REPO)
    require(digest((REPO / QUAL_PLAN).read_bytes()) == QUAL_HASH,
            "STOP: predeclared qualification plan mismatch")
    plan = read_json(REPO / "configs/pre_freeze/paligemma_t4_runtime.v1.json")
    canonical = json.dumps(plan, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False).encode()
    require(digest(canonical) == PLAN_HASH and plan["model_id"] == MODEL
            and plan["model_revision"] == REVISION and len(plan["snapshot_files"]) == 14,
            "STOP: pinned plan mismatch")
    write_new(SETUP / "execution_pin.json", {"execution_commit": BASE, "base_sha": BASE_SHA,
              "qualification_plan_sha256": QUAL_HASH, "plan_sha256": PLAN_HASH,
              "requirements_sha256": digest((REPO / "requirements-paligemma-t4.txt").read_bytes())})

    PHASE = "FROZEN_SUITE"
    command([sys.executable, "scripts/w2_paligemma_external_gate.py", "--verify-suite"],
            "verify_suite", cwd=REPO,
            failure="STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED: frozen gate bytes missing or changed")
    PHASE = "SOFTWARE_INSTALL"
    bootstrap = WORK / "bootstrap"
    require(not bootstrap.exists(), "STOP: bootstrap location exists; no overwrite")
    command([sys.executable, "-m", "pip", "--isolated", "install", "--no-cache-dir",
             "--no-deps", "--only-binary=:all:", "--ignore-installed",
             "--target", bootstrap, "uv==0.8.22"], "bootstrap_uv")
    uv = bootstrap / "bin/uv"
    _, uv_version = command([uv, "--version"], "bootstrap_uv_version")
    require(re.fullmatch(r"uv 0\.8\.22(?: \([^\r\n()]+\))?", uv_version.strip()) is not None,
            "STOP: bootstrap requires exactly uv 0.8.22")
    command([uv, "--no-config", "python", "install", "3.11.11"], "python_install")
    command([uv, "--no-config", "venv", "--managed-python", "--python", "3.11.11", "--seed", ENV], "runtime_venv")
    command([PYTHON, "-m", "pip", "--isolated", "install", "--no-cache-dir", "--only-binary=:all:",
             "-r", "requirements-paligemma-t4.txt"], "install_pins", cwd=REPO,
            failure="STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED: exact package/wheel installation failed")
    command([PYTHON, "-m", "pip", "--isolated", "check"], "pip_check", cwd=REPO,
            failure="STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED: pip check failed")
    command([PYTHON, "-m", "pip", "--isolated", "list", "--format=json"], "installed_distributions", cwd=REPO)
    command([PYTHON, "-m", "pip", "--isolated", "freeze", "--all"], "pip_freeze", cwd=REPO)
    PHASE = "SOFTWARE_AND_PROCESS_GPU"
    probe = r'''import json
from safeshift.runners.paligemma_snapshot import load_plan, network_denied, require_offline_env
from safeshift.runners.paligemma import software_versions, probe_hardware
require_offline_env()
with network_denied():
    actual = software_versions()
    if actual != load_plan()["software"]:
        raise RuntimeError("STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED: exact pins mismatch")
    import torch
    hardware = probe_hardware(torch)
    print(json.dumps({"software": actual, "hardware": hardware}, indent=2))
'''
    _, probe_output = command([PYTHON, "-c", probe], "process_probe", cwd=REPO, env=child_env(offline=True))
    write_new(SETUP / "software_hardware.json", json.loads(probe_output))

    PHASE = "PROVISION"
    cache = REPO / CACHE
    require(not cache.exists(), "STOP: cache is not fresh; no repair or overwrite")
    provision_env = child_env()
    provision_env["HF_TOKEN"] = hf_token
    try:
        command([PYTHON, "scripts/provision_paligemma_snapshot.py", "--provision", "--cache-dir", CACHE,
                 "--manifest", MANIFESTS + "/provision.json"], "provision", cwd=REPO, env=provision_env)
    finally:
        provision_env.clear()
        hf_token = None
    PHASE = "VERIFY_SNAPSHOT_ONLINE"
    command([PYTHON, "scripts/provision_paligemma_snapshot.py", "--verify-only", "--cache-dir", CACHE,
             "--manifest", MANIFESTS + "/verify_online.json"], "verify_online", cwd=REPO, env=child_env(offline=True))
    verification = read_json(REPO / MANIFESTS / "verify_online.json")
    require(verification["local_bytes_verified"] is True and len(verification["files"]) == 14
            and verification["revision"] == REVISION, "STOP: snapshot verification not established")
    clean_checkout()
    write_new(SETUP / "online_complete.json", {"snapshot_verified": True, "execution_commit": BASE,
              "finished_at_utc": datetime.now(timezone.utc).isoformat()})
    ONLINE_DONE = True
    PHASE = "AWAITING_OWNER_INTERNET_OFF"
    print("SNAPSHOT_VERIFIED=true; PROVISION COMPLETE — MANUAL ACTION REQUIRED")
except Exception as exc:
    report = {**boundaries(), "status": "STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED", "failed_phase": PHASE,
              "reason": sanitize(str(exc)), "runtime_executed": False}
    write_new(SETUP / "failure.json", report)
    build_bundle(report)
    raise RuntimeError(report["reason"]) from None
finally:
    hf_token = None


## PROVISION COMPLETE — MANUAL INTERNET-OFF BARRIER

Turn Kaggle Internet OFF without restarting. Set OWNER_ATTEST_INTERNET_OFF=True in the next cell. Run All stops while it is False. The boolean records owner attestation, not a network measurement; runtime additionally denies sockets and requires offline Hub/Transformers.

The fixed attempt verifies local snapshot bytes, loads once, calls eight times and exports d9r13_result_bundle.zip. Raw native output is exclusively written, flushed/fsynced, reread and SHA/size verified before parsing. No retry, repair, label correction or extra cases. Timing includes audits and persistence.


In [ ]:
OWNER_ATTEST_INTERNET_OFF = False
require(OWNER_ATTEST_INTERNET_OFF is True,
        "STOP: turn venue Internet OFF, set OWNER_ATTEST_INTERNET_OFF = True, then continue this cell")
require(ONLINE_DONE is True and (SETUP / "online_complete.json").is_file(),
        "STOP: verified online phase must finish in this session")
require(not (SETUP / "runtime_attempt.json").exists(),
        "STOP: runtime attempt already started; no retry or new attempt ID")
write_new(SETUP / "runtime_attempt.json", {"owner_attested_internet_off": True,
          "attested_at_utc": datetime.now(timezone.utc).isoformat(), "run_id": RUN_ID})
runtime_env = child_env(offline=True)
try:
    PHASE = "OFFLINE_VERIFY"
    clean_checkout()
    command([PYTHON, "scripts/provision_paligemma_snapshot.py", "--verify-only", "--cache-dir", CACHE,
             "--manifest", MANIFESTS + "/verify_offline.json"], "verify_offline", cwd=REPO, env=runtime_env)
    offline = read_json(REPO / MANIFESTS / "verify_offline.json")
    online = read_json(REPO / MANIFESTS / "verify_online.json")
    require(offline["local_bytes_verified"] is True and offline["files"] == online["files"]
            and len(offline["files"]) == 14, "STOP: offline snapshot mismatch")
    PHASE = "FROZEN_EXTERNAL_GATE_EVIDENCE_COLLECTION"
    clean_checkout()
    returncode, _ = command([PYTHON, "scripts/w2_paligemma_external_gate.py",
                            "--expected-commit", BASE, "--cache-dir", CACHE, "--venue-internet-off"],
                           "external_gate", cwd=REPO, env=runtime_env, check=False)
    PHASE = "EVIDENCE_REVIEW"
    report = review_runtime(returncode)
    clean_checkout()
except Exception as exc:
    report = {**boundaries(), "status": "STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED",
              "failed_phase": PHASE, "reason": sanitize(str(exc)), "runtime_attempt_recorded": True}
    write_new(SETUP / "failure.json", report)
    build_bundle(report)
    raise RuntimeError(report["reason"]) from None
else:
    build_bundle(report)
    print("EVIDENCE_COLLECTION_COMPLETE")
    print(ZIP)
finally:
    runtime_env.clear()
    SECRET_VALUES.clear()


Submit d9r13_result_bundle.zip to the Research Lead. It includes all eight native outputs, raw SHA references, parser status/diagnostics, predicted and both GT boxes, frozen images, per-case metrics, swap tracking, the human-review template, and unchanged D9R11 negative evidence.

To finalize PASS/FAIL, retain this exact clean execution checkout and runtime artifacts. Review the eight images/predictions; copy the template to `data/processed/d9r13_human_review.json` and fill each status with NO_GIANT or GIANT plus reviewer and rationale. Do not edit raw, GT, or the original report. Run the following offline command with the notebook's exact BASE commit (no model/GPU access):

```text
<runtime python> scripts/w2_paligemma_external_gate.py --expected-commit <BASE> --review-file data/processed/d9r13_human_review.json
```

This revalidates artifacts, reparses raw and calls the unchanged evaluator. It writes exclusive `final_gate_review.json` and `final_gate_review.sha256.json` alongside summary.json; submit both with the original ZIP. For review on another machine, checkout BASE and restore the bundle's runtime/ contents to `data/processed/external_gate/w2_paligemma/paligemma-d9r13-external-gate-01/`. Dependencies for this review path are CPU-only; no snapshot/model load is needed.

FAIL remains FAIL even with NO_GIANT reviews. PASS does not promote PaliGemma: Research Lead still considers D9R11 hallucination. Classification remains PENDING_QUALIFICATION, PaliGemma BACKUP_1, four primaries, protocol freeze BLOCKED. No InspecSafe or additional synthetic run.
